# Local vs global training: representation structure comparison

Comparing how local (Forward-Forward) and global (Backpropagation) training
objectives shape internal representations, with confounder controls and
statistical tests across seeds.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

# Load all available datasets
datasets = {}
for name in ['mnist', 'cifar10']:
    p = Path(f'results/{name}.json')
    if p.exists():
        with open(p) as f:
            datasets[name] = json.load(f)

# Load scaling results if available
scaling_path = Path('results/scaling.json')
scaling = json.load(open(scaling_path)) if scaling_path.exists() else None

FF_C, BP_C = '#2563eb', '#dc2626'
SEED_ALPHA = 0.45

print(f"Datasets loaded: {', '.join(datasets.keys())}")
if scaling:
    print(f"Scaling study: {len(scaling)} configs")

In [ ]:
# --- Helper functions ---

def avg_per_layer(runs, key, metric):
    """Mean and std of a per-layer metric across seeds."""
    n_layers = len(runs[0][key])
    means, stds, per_seed = [], [], []
    for l in range(n_layers):
        vals = [r[key][l][metric] for r in runs]
        means.append(np.mean(vals)); stds.append(np.std(vals))
        per_seed.append(vals)
    return means, stds, per_seed

def avg_metric(runs, key, metric):
    """Mean of a metric averaged across layers, per seed. Returns (mean, std, per_seed_vals)."""
    vals = [np.mean([m[metric] for m in r[key]]) for r in runs]
    return np.mean(vals), np.std(vals), vals

def paired_test(ff_vals, bp_vals):
    """Paired t-test (or Wilcoxon if n >= 6). Returns (statistic, p-value, test_name)."""
    n = len(ff_vals)
    if n < 2:
        return 0.0, 1.0, 'n/a'
    if n >= 6:
        stat, p = stats.wilcoxon(ff_vals, bp_vals)
        return stat, p, 'wilcoxon'
    stat, p = stats.ttest_rel(ff_vals, bp_vals)
    return stat, p, f'paired-t(df={n-1})'

def bar_with_seeds(ax, x_pos, vals_list, width, color, label, seed_vals=None):
    """Bar chart with individual seed points overlaid."""
    bars = ax.bar(x_pos, vals_list, width, color=color, label=label, zorder=3, alpha=0.85)
    if seed_vals is not None:
        for i, sv in enumerate(seed_vals):
            jitter = np.random.default_rng(42).uniform(-width*0.2, width*0.2, len(sv))
            ax.scatter(x_pos[i] + jitter, sv, color=color, s=18, zorder=4,
                       alpha=SEED_ALPHA, edgecolors='white', linewidths=0.5)
    return bars

def fmt_p(p):
    if p < 0.001: return 'p<.001'
    if p < 0.01: return f'p={p:.3f}'
    if p < 0.05: return f'p={p:.2f}'
    return f'p={p:.2f} (ns)'

## Summary statistics with paired tests

Paired across seeds. With n=3, these are paired t-tests (df=1 or 2). Take p-values
as directional evidence, not hard thresholds.

In [ ]:
for ds_name, ds in datasets.items():
    runs = ds['runs']
    config = ds['config']
    n_seeds = len(runs)

    print(f"\n{'='*78}")
    print(f"  {ds_name.upper()}  |  {config['layers']}x{config['hidden']} MLP  |  "
          f"{config['epochs']} epochs  |  {n_seeds} seeds")
    print(f"{'='*78}")

    ff_accs = [r['ff_accuracy'] for r in runs]
    bp_accs = [r['bp_accuracy'] for r in runs]
    _, p_acc, test_acc = paired_test(ff_accs, bp_accs)
    print(f"  FF accuracy: {np.mean(ff_accs):.4f} +/- {np.std(ff_accs):.4f}")
    print(f"  BP accuracy: {np.mean(bp_accs):.4f} +/- {np.std(bp_accs):.4f}")
    print(f"  delta: {np.mean(ff_accs) - np.mean(bp_accs):+.4f}  [{test_acc}, {fmt_p(p_acc)}]")

    pixel_vals = [r['pixel_probe'] for r in runs]
    print(f"  pixel probe: {np.mean(pixel_vals):.4f} +/- {np.std(pixel_vals):.4f}")

    print(f"\n  {'Metric':<22} {'FF':>16} {'BP':>16} {'Delta':>8}  {'Test':>20}")
    print(f"  {'-'*86}")

    metrics = ['probe_acc', 'probe_acc_masked', 'sparsity', 'dead_frac',
               'eff_rank', 'correlation', 'polysemanticity']
    for m in metrics:
        ff_mean, ff_std, ff_v = avg_metric(runs, 'ff_metrics', m)
        bp_mean, bp_std, bp_v = avg_metric(runs, 'bp_metrics', m)
        _, p, tname = paired_test(ff_v, bp_v)
        d = ff_mean - bp_mean
        print(f"  {m:<22} {ff_mean:.4f}+/-{ff_std:.4f} {bp_mean:.4f}+/-{bp_std:.4f} {d:+.4f}  {tname} {fmt_p(p)}")

    # Pruning at 90%
    ff_prune_raw = [r['ff_pruning'].get('0.9', 0) for r in runs]
    bp_prune_raw = [r['bp_pruning'].get('0.9', 0) for r in runs]
    ff_prune_live = [r['ff_pruning_live'].get('0.9', 0) for r in runs]
    bp_prune_live = [r['bp_pruning_live'].get('0.9', 0) for r in runs]
    _, p_raw, t_raw = paired_test(ff_prune_raw, bp_prune_raw)
    _, p_live, t_live = paired_test(ff_prune_live, bp_prune_live)
    print(f"  {'prune@90% (raw)':<22} {np.mean(ff_prune_raw):.4f}+/-{np.std(ff_prune_raw):.4f} "
          f"{np.mean(bp_prune_raw):.4f}+/-{np.std(bp_prune_raw):.4f} "
          f"{np.mean(ff_prune_raw)-np.mean(bp_prune_raw):+.4f}  {t_raw} {fmt_p(p_raw)}")
    print(f"  {'prune@90% (live)':<22} {np.mean(ff_prune_live):.4f}+/-{np.std(ff_prune_live):.4f} "
          f"{np.mean(bp_prune_live):.4f}+/-{np.std(bp_prune_live):.4f} "
          f"{np.mean(ff_prune_live)-np.mean(bp_prune_live):+.4f}  {t_live} {fmt_p(p_live)}")

    # Ablation
    ff_abl = [r['ff_ablation']['mean'] for r in runs]
    bp_abl = [r['bp_ablation']['mean'] for r in runs]
    _, p_abl, t_abl = paired_test(ff_abl, bp_abl)
    print(f"  {'ablation (mean drop)':<22} {np.mean(ff_abl):.4f}+/-{np.std(ff_abl):.4f} "
          f"{np.mean(bp_abl):.4f}+/-{np.std(bp_abl):.4f} "
          f"{np.mean(ff_abl)-np.mean(bp_abl):+.4f}  {t_abl} {fmt_p(p_abl)}")

## Confounder control variants (BP+norm, BP+overlay)

Phase 0.5 ablation: BP+norm isolates the normalization effect, BP+overlay isolates label injection.
Both are trained by default and included in the result JSON.

In [ ]:
CONTROL_VARIANTS = [
    ('bp_norm', 'BP+norm'),
    ('bp_overlay', 'BP+overlay'),
]

for ds_name, ds in datasets.items():
    runs = ds['runs']
    available = [(k, lab) for k, lab in CONTROL_VARIANTS if f'{k}_accuracy' in runs[0]]
    if not available:
        print(f"{ds_name.upper()}: no confounder-control variants in data (run with --variants ff bp bp_norm bp_overlay)")
        continue

    print(f"\n{'='*78}")
    print(f"  {ds_name.upper()} CONFOUNDER CONTROLS")
    print(f"{'='*78}")

    # Reference baselines
    ff_acc = np.mean([r['ff_accuracy'] for r in runs])
    bp_acc = np.mean([r['bp_accuracy'] for r in runs])
    print(f"\n  Reference:  FF={ff_acc:.4f}  BP={bp_acc:.4f}")

    for vname, vlabel in available:
        accs = [r[f'{vname}_accuracy'] for r in runs]
        print(f"\n  {vlabel} accuracy: {np.mean(accs):.4f} +/- {np.std(accs):.4f}")

        key = f'{vname}_metrics'
        if key not in runs[0]:
            continue

        metrics = ['probe_acc', 'probe_acc_masked', 'sparsity', 'dead_frac',
                   'eff_rank', 'polysemanticity']
        print(f"  {'Metric':<22} {vlabel:>12} {'BP':>12} {'FF':>12}")
        print(f"  {'-'*60}")
        for m in metrics:
            v_mean, v_std, _ = avg_metric(runs, key, m)
            bp_mean, _, _ = avg_metric(runs, 'bp_metrics', m)
            ff_mean, _, _ = avg_metric(runs, 'ff_metrics', m)
            print(f"  {m:<22} {v_mean:>12.4f} {bp_mean:>12.4f} {ff_mean:>12.4f}")

        # Pruning comparison
        v_prune = [r.get(f'{vname}_pruning_live', {}).get('0.9', 0) for r in runs]
        bp_prune = [r['bp_pruning_live'].get('0.9', 0) for r in runs]
        if any(v > 0 for v in v_prune):
            print(f"  {'prune@90% (live)':<22} {np.mean(v_prune):>12.4f} {np.mean(bp_prune):>12.4f}")

## Per-layer metrics (both datasets)

Each panel shows per-layer means with individual seed points overlaid.
With n=3, the scatter gives a more honest picture than error bars alone.

In [ ]:
n_ds = len(datasets)
fig, axes = plt.subplots(4, n_ds, figsize=(8 * n_ds, 16), facecolor='white',
                         squeeze=False)
fig.suptitle('Per-layer representation metrics', fontsize=16, fontweight='bold', y=0.98)
W = 0.32

panel_specs = [
    ('probe_acc_masked', 'Probe accuracy (label-masked)', 'Accuracy'),
    ('sparsity', 'Activation sparsity', 'Fraction near-zero'),
    ('polysemanticity', 'Polysemanticity', 'Classes per neuron'),
    ('dead_frac', 'Dead neuron fraction', 'Fraction dead'),
]

for col, (ds_name, ds) in enumerate(datasets.items()):
    runs = ds['runs']
    n_layers = len(runs[0]['ff_metrics'])
    x = np.arange(n_layers)
    layer_labels = [f'L{i}' for i in range(n_layers)]

    # Pixel probe baseline (for reference line on probe panel)
    pixel_mean = np.mean([r['pixel_probe'] for r in runs])

    for row, (metric, title, ylabel) in enumerate(panel_specs):
        ax = axes[row, col]
        ff_m, ff_s, ff_seeds = avg_per_layer(runs, 'ff_metrics', metric)
        bp_m, bp_s, bp_seeds = avg_per_layer(runs, 'bp_metrics', metric)

        bar_with_seeds(ax, x - W/2, ff_m, W, FF_C, 'FF', ff_seeds)
        bar_with_seeds(ax, x + W/2, bp_m, W, BP_C, 'BP', bp_seeds)

        # Add pixel probe reference line on probe accuracy panels
        if metric == 'probe_acc_masked':
            ax.axhline(pixel_mean, color='#888', ls='--', lw=1.2, zorder=2)
            ax.text(n_layers - 0.5, pixel_mean + 0.008, f'pixel baseline {pixel_mean:.2%}',
                    ha='right', fontsize=8, color='#888')

        if metric == 'polysemanticity':
            ax.text(0.02, 0.02, 'Lower = more monosemantic', transform=ax.transAxes,
                    fontsize=8, color='#888', style='italic')

        ax.set_ylabel(ylabel)
        ax.set_title(f'{title} ({ds_name.upper()})', fontweight='bold')
        ax.set_xticks(x); ax.set_xticklabels(layer_labels)
        ax.legend(framealpha=0.9, fontsize=9)
        ax.grid(axis='y', alpha=0.3, zorder=0)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('fig1_per_layer.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved fig1_per_layer.png')

## Confounder controls: raw vs controlled

The strongest methodological argument in this project. Left column shows raw
metrics, right column shows the controlled version. The difference between
columns is the confounder's magnitude.

In [ ]:
fig, axes = plt.subplots(2, n_ds, figsize=(8 * n_ds, 10), facecolor='white',
                         squeeze=False)
fig.suptitle('Confounder controls: raw vs controlled', fontsize=16,
             fontweight='bold', y=0.98)
W = 0.18

for col, (ds_name, ds) in enumerate(datasets.items()):
    runs = ds['runs']
    n_layers = len(runs[0]['ff_metrics'])
    x = np.arange(n_layers)
    layer_labels = [f'L{i}' for i in range(n_layers)]

    # Row 0: Probe accuracy (raw vs label-masked)
    ax = axes[0, col]
    ff_raw_m, _, ff_raw_s = avg_per_layer(runs, 'ff_metrics', 'probe_acc')
    ff_mask_m, _, ff_mask_s = avg_per_layer(runs, 'ff_metrics', 'probe_acc_masked')
    bp_raw_m, _, bp_raw_s = avg_per_layer(runs, 'bp_metrics', 'probe_acc')
    bp_mask_m, _, bp_mask_s = avg_per_layer(runs, 'bp_metrics', 'probe_acc_masked')

    offsets = [-1.5*W, -0.5*W, 0.5*W, 1.5*W]
    ax.bar(x + offsets[0], ff_raw_m, W, color=FF_C, alpha=0.45, label='FF raw', zorder=3)
    ax.bar(x + offsets[1], ff_mask_m, W, color=FF_C, alpha=0.95, label='FF masked', zorder=3)
    ax.bar(x + offsets[2], bp_raw_m, W, color=BP_C, alpha=0.45, label='BP raw', zorder=3)
    ax.bar(x + offsets[3], bp_mask_m, W, color=BP_C, alpha=0.95, label='BP masked', zorder=3)

    lo = min(min(ff_mask_m), min(bp_mask_m))
    ax.set_ylim(max(0, lo - 0.04), 1.005)
    ax.set_ylabel('Probe accuracy')
    ax.set_title(f'Label overlay confounder ({ds_name.upper()})', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(layer_labels)
    ax.legend(fontsize=8, ncol=2, framealpha=0.9)
    ax.grid(axis='y', alpha=0.3, zorder=0)

    # Row 1: Pruning curves (raw vs live-only)
    ax = axes[1, col]
    levels = sorted(runs[0]['ff_pruning_live'].keys(), key=float)
    sp = [float(s) for s in levels]

    for key, color, ls, label in [
        ('ff_pruning', FF_C, '--', 'FF raw'),
        ('ff_pruning_live', FF_C, '-', 'FF live-only'),
        ('bp_pruning', BP_C, '--', 'BP raw'),
        ('bp_pruning_live', BP_C, '-', 'BP live-only'),
    ]:
        curves = np.array([[r[key][s] for s in levels] for r in runs])
        mean = curves.mean(0)
        alpha = 0.5 if '--' in ls else 1.0
        ax.plot(sp, mean, ls, color=color, lw=2, ms=6, marker='o' if color == FF_C else 's',
                label=label, alpha=alpha, zorder=3)
        if ls == '-':
            std = curves.std(0)
            ax.fill_between(sp, mean - std, np.minimum(mean + std, 1.0),
                            alpha=0.1, color=color)

    ax.set_xlabel('Fraction pruned')
    ax.set_ylabel('Probe accuracy after pruning')
    ax.set_title(f'Dead neuron confounder ({ds_name.upper()})', fontweight='bold')
    ax.set_xlim(-0.02, 0.92)
    ax.legend(fontsize=9, framealpha=0.9)
    ax.grid(alpha=0.3, zorder=0)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('fig2_confounders.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved fig2_confounders.png')

## Ablation effects and effective rank

Ablation: mean probe accuracy drop when individual live neurons are zeroed.
Near-zero mean = redundantly distributed information. Large max = localized,
causally significant neurons.

Effective rank: exponential of the entropy of the singular value spectrum.
Higher = more dimensions carrying information.

In [ ]:
fig, axes = plt.subplots(2, n_ds, figsize=(8 * n_ds, 10), facecolor='white',
                         squeeze=False)
fig.suptitle('Causal structure and representation geometry', fontsize=16,
             fontweight='bold', y=0.98)
W = 0.32

for col, (ds_name, ds) in enumerate(datasets.items()):
    runs = ds['runs']
    n_layers = len(runs[0]['ff_metrics'])
    x = np.arange(n_layers)
    layer_labels = [f'L{i}' for i in range(n_layers)]

    # Row 0: Ablation effects (bar chart: mean and max drop)
    ax = axes[0, col]
    ff_mean_drops = [r['ff_ablation']['mean'] for r in runs]
    bp_mean_drops = [r['bp_ablation']['mean'] for r in runs]
    ff_max_drops = [r['ff_ablation']['max'] for r in runs]
    bp_max_drops = [r['bp_ablation']['max'] for r in runs]

    categories = ['Mean drop', 'Max drop']
    cx = np.arange(len(categories))
    ff_vals = [np.mean(ff_mean_drops), np.mean(ff_max_drops)]
    bp_vals = [np.mean(bp_mean_drops), np.mean(bp_max_drops)]
    ff_seeds_abl = [ff_mean_drops, ff_max_drops]
    bp_seeds_abl = [bp_mean_drops, bp_max_drops]

    bar_with_seeds(ax, cx - W/2, ff_vals, W, FF_C, 'FF', ff_seeds_abl)
    bar_with_seeds(ax, cx + W/2, bp_vals, W, BP_C, 'BP', bp_seeds_abl)

    ax.set_ylabel('Probe accuracy drop')
    ax.set_title(f'Single-neuron ablation effect ({ds_name.upper()})', fontweight='bold')
    ax.set_xticks(cx); ax.set_xticklabels(categories)
    ax.legend(framealpha=0.9)
    ax.grid(axis='y', alpha=0.3, zorder=0)
    ax.text(0.02, 0.95, 'Last layer, per live neuron', transform=ax.transAxes,
            fontsize=8, color='#888', style='italic', va='top')

    # Row 1: Effective rank per layer
    ax = axes[1, col]
    ff_m, ff_s, ff_seeds = avg_per_layer(runs, 'ff_metrics', 'eff_rank')
    bp_m, bp_s, bp_seeds = avg_per_layer(runs, 'bp_metrics', 'eff_rank')

    bar_with_seeds(ax, x - W/2, ff_m, W, FF_C, 'FF', ff_seeds)
    bar_with_seeds(ax, x + W/2, bp_m, W, BP_C, 'BP', bp_seeds)

    ax.set_ylabel('Effective rank')
    ax.set_title(f'Effective rank ({ds_name.upper()})', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(layer_labels)
    ax.legend(framealpha=0.9)
    ax.grid(axis='y', alpha=0.3, zorder=0)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('fig3_ablation_rank.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved fig3_ablation_rank.png')

## Scaling study

Do the representation structure differences hold, shrink, or grow with model size?
Five configurations from 200K to 8M parameters.

In [ ]:
if scaling is None:
    print("No scaling results found (results/scaling.json). Skipping.")
else:
    def scale_get(r, key, metric):
        return np.mean([m[metric] for m in r[key]])

    labels = [f"{r['config']}\n{r['n_params']/1e6:.1f}M" for r in scaling]
    x = np.arange(len(scaling))
    W = 0.32

    fig, axes = plt.subplots(2, 3, figsize=(18, 11), facecolor='white')
    fig.suptitle('Scaling study: local vs global across model size',
                 fontsize=16, fontweight='bold', y=0.98)

    # Panel 1: Task accuracy
    ax = axes[0, 0]
    ff_v = [r['ff_accuracy'] for r in scaling]
    bp_v = [r['bp_accuracy'] for r in scaling]
    ax.bar(x - W/2, ff_v, W, color=FF_C, label='FF', zorder=3)
    ax.bar(x + W/2, bp_v, W, color=BP_C, label='BP', zorder=3)
    ax.set_ylabel('Test accuracy'); ax.set_title('Task accuracy', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
    lo = min(ff_v + bp_v); ax.set_ylim(max(0, lo - 0.05), 1.005)
    ax.legend(framealpha=0.9); ax.grid(axis='y', alpha=0.3, zorder=0)

    # Panel 2: Probe accuracy (label-masked)
    ax = axes[0, 1]
    ff_v = [scale_get(r, 'ff_metrics', 'probe_acc_masked') for r in scaling]
    bp_v = [scale_get(r, 'bp_metrics', 'probe_acc_masked') for r in scaling]
    ax.bar(x - W/2, ff_v, W, color=FF_C, label='FF', zorder=3)
    ax.bar(x + W/2, bp_v, W, color=BP_C, label='BP', zorder=3)
    ax.set_ylabel('Probe accuracy (avg)'); ax.set_title('Probe accuracy (label-masked)', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
    lo = min(ff_v + bp_v); ax.set_ylim(max(0, lo - 0.03), 1.005)
    ax.legend(framealpha=0.9); ax.grid(axis='y', alpha=0.3, zorder=0)

    # Panel 3: Polysemanticity
    ax = axes[0, 2]
    ff_v = [scale_get(r, 'ff_metrics', 'polysemanticity') for r in scaling]
    bp_v = [scale_get(r, 'bp_metrics', 'polysemanticity') for r in scaling]
    ax.bar(x - W/2, ff_v, W, color=FF_C, label='FF', zorder=3)
    ax.bar(x + W/2, bp_v, W, color=BP_C, label='BP', zorder=3)
    ax.set_ylabel('Classes per neuron'); ax.set_title('Polysemanticity', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
    ax.legend(framealpha=0.9); ax.grid(axis='y', alpha=0.3, zorder=0)

    # Panel 4: Pruning@90% (live)
    ax = axes[1, 0]
    ff_v = [r['ff_pruning_live'].get('0.9', 0) for r in scaling]
    bp_v = [r['bp_pruning_live'].get('0.9', 0) for r in scaling]
    ax.bar(x - W/2, ff_v, W, color=FF_C, label='FF', zorder=3)
    ax.bar(x + W/2, bp_v, W, color=BP_C, label='BP', zorder=3)
    ax.set_ylabel('Probe accuracy'); ax.set_title('Pruning@90% (live neurons)', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
    lo = min(ff_v + bp_v); ax.set_ylim(max(0, lo - 0.05), 1.005)
    ax.legend(framealpha=0.9); ax.grid(axis='y', alpha=0.3, zorder=0)

    # Panel 5: Delta trends across scale
    ax = axes[1, 1]
    probe_d = [scale_get(r, 'ff_metrics', 'probe_acc_masked') -
               scale_get(r, 'bp_metrics', 'probe_acc_masked') for r in scaling]
    poly_d = [scale_get(r, 'ff_metrics', 'polysemanticity') -
              scale_get(r, 'bp_metrics', 'polysemanticity') for r in scaling]
    prune_d = [r['ff_pruning_live'].get('0.9', 0) -
               r['bp_pruning_live'].get('0.9', 0) for r in scaling]
    ax.plot(x, probe_d, 'o-', color='#16a34a', lw=2, ms=8, label='Probe (masked)')
    ax.plot(x, poly_d, 's-', color='#9333ea', lw=2, ms=8, label='Polysemanticity')
    ax.plot(x, prune_d, '^-', color='#ea580c', lw=2, ms=8, label='Pruning (live)')
    ax.axhline(0, color='black', lw=0.5, ls='--')
    ax.set_ylabel('FF - BP (positive = FF higher)')
    ax.set_title('Delta vs scale (controlled metrics)', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
    ax.legend(fontsize=9, framealpha=0.9); ax.grid(alpha=0.3)

    # Panel 6: Effective rank across scale
    ax = axes[1, 2]
    ff_v = [scale_get(r, 'ff_metrics', 'eff_rank') for r in scaling]
    bp_v = [scale_get(r, 'bp_metrics', 'eff_rank') for r in scaling]
    ax.bar(x - W/2, ff_v, W, color=FF_C, label='FF', zorder=3)
    ax.bar(x + W/2, bp_v, W, color=BP_C, label='BP', zorder=3)
    ax.set_ylabel('Effective rank (avg)'); ax.set_title('Effective rank vs scale', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
    ax.legend(framealpha=0.9); ax.grid(axis='y', alpha=0.3, zorder=0)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('fig4_scaling.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print('Saved fig4_scaling.png')

## Computed findings

The cell below generates a summary from actual values, so the text stays
in sync with the data.

In [ ]:
for ds_name, ds in datasets.items():
    runs = ds['runs']
    config = ds['config']
    n_seeds = len(runs)

    ff_acc_m = np.mean([r['ff_accuracy'] for r in runs])
    bp_acc_m = np.mean([r['bp_accuracy'] for r in runs])
    pixel_m = np.mean([r['pixel_probe'] for r in runs])

    ff_probe_m, _, ff_probe_v = avg_metric(runs, 'ff_metrics', 'probe_acc_masked')
    bp_probe_m, _, bp_probe_v = avg_metric(runs, 'bp_metrics', 'probe_acc_masked')
    _, p_probe, t_probe = paired_test(ff_probe_v, bp_probe_v)
    probe_gap = ff_probe_m - bp_probe_m

    ff_poly_m, _, ff_poly_v = avg_metric(runs, 'ff_metrics', 'polysemanticity')
    bp_poly_m, _, bp_poly_v = avg_metric(runs, 'bp_metrics', 'polysemanticity')

    ff_dead_m, _, _ = avg_metric(runs, 'ff_metrics', 'dead_frac')
    bp_dead_m, _, _ = avg_metric(runs, 'bp_metrics', 'dead_frac')

    ff_rank_m, _, _ = avg_metric(runs, 'ff_metrics', 'eff_rank')
    bp_rank_m, _, _ = avg_metric(runs, 'bp_metrics', 'eff_rank')

    ff_prune_live_90 = np.mean([r['ff_pruning_live'].get('0.9', 0) for r in runs])
    bp_prune_live_90 = np.mean([r['bp_pruning_live'].get('0.9', 0) for r in runs])
    ff_prune_raw_90 = np.mean([r['ff_pruning'].get('0.9', 0) for r in runs])
    bp_prune_raw_90 = np.mean([r['bp_pruning'].get('0.9', 0) for r in runs])

    ff_abl_m = np.mean([r['ff_ablation']['mean'] for r in runs])
    bp_abl_m = np.mean([r['bp_ablation']['mean'] for r in runs])
    ff_abl_max = np.mean([r['ff_ablation']['max'] for r in runs])
    bp_abl_max = np.mean([r['bp_ablation']['max'] for r in runs])

    print(f"\n{'='*70}")
    print(f"FINDINGS: {ds_name.upper()} ({config['layers']}x{config['hidden']}, "
          f"{config['epochs']} epochs, {n_seeds} seeds)")
    print(f"{'='*70}")

    print(f"\nTask accuracy: FF {ff_acc_m:.1%} vs BP {bp_acc_m:.1%} "
          f"(BP wins by {bp_acc_m - ff_acc_m:.1%})")

    print(f"\nLinear separability: FF probe (masked) {ff_probe_m:.1%} vs BP {bp_probe_m:.1%}, "
          f"gap = {probe_gap:+.1%} [{t_probe}, {fmt_p(p_probe)}]. "
          f"Pixel baseline is {pixel_m:.1%}, so both models learn structure "
          f"beyond raw input.")

    print(f"\nPruning robustness: at 90% live neurons pruned, "
          f"FF retains {ff_prune_live_90:.1%} vs BP {bp_prune_live_90:.1%}. "
          f"Raw (uncorrected) pruning: FF {ff_prune_raw_90:.1%} vs BP {bp_prune_raw_90:.1%}. "
          f"Controlling for dead neurons reduces the gap from "
          f"{ff_prune_raw_90 - bp_prune_raw_90:+.1%} to "
          f"{ff_prune_live_90 - bp_prune_live_90:+.1%} but does not eliminate it.")

    print(f"\nPolysemanticity: FF {ff_poly_m:.2f} vs BP {bp_poly_m:.2f} classes/neuron. "
          f"{'BP' if bp_poly_m < ff_poly_m else 'FF'} neurons are more selective individually.")

    print(f"\nEffective rank: FF {ff_rank_m:.1f} vs BP {bp_rank_m:.1f}. "
          f"{'BP' if bp_rank_m > ff_rank_m else 'FF'} uses more dimensions, "
          f"consistent with {'lower' if ff_dead_m > bp_dead_m else 'higher'} dead fraction "
          f"(FF {ff_dead_m:.1%} vs BP {bp_dead_m:.1%}).")

    print(f"\nAblation: mean single-neuron drop FF {ff_abl_m:.4f} vs BP {bp_abl_m:.4f}. "
          f"Max drop FF {ff_abl_max:.4f} vs BP {bp_abl_max:.4f}. "
          f"{'BP' if bp_abl_max > ff_abl_max else 'FF'} has more causally concentrated neurons.")